<a href="https://colab.research.google.com/github/76213869-sketch/Sem7_MLP2/blob/main/Prueba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
# FASE 6: PREDICCIÓN MANUAL — PRUEBA EL MODELO
# ============================================================
import numpy as np
import pandas as pd

print("=" * 60)
print("       PREDICTOR DE DESERCIÓN ESTUDIANTIL")
print("=" * 60)
print()

# =============================
# INGRESA LOS DATOS DEL ESTUDIANTE
# =============================

# ── ACADÉMICOS 1er semestre ──────────────────────────────────
cu1_enrolled  = float(input("Unidades 1er sem (matriculadas)  [ej: 6]: "))
cu1_approved  = float(input("Unidades 1er sem (aprobadas)     [ej: 5]: "))
cu1_grade     = float(input("Nota promedio 1er sem            [0-20, ej: 13.5]: "))

# ── ACADÉMICOS 2do semestre ──────────────────────────────────
cu2_enrolled  = float(input("Unidades 2do sem (matriculadas)  [ej: 6]: "))
cu2_approved  = float(input("Unidades 2do sem (aprobadas)     [ej: 4]: "))
cu2_grade     = float(input("Nota promedio 2do sem            [0-20, ej: 11.0]: "))

# ── FINANCIERO ───────────────────────────────────────────────
tuition_ok    = float(input("¿Matrícula al día?               [1=Sí, 0=No]: "))
debtor        = float(input("¿Deudor?                         [1=Sí, 0=No]: "))
scholarship   = float(input("¿Becario?                        [1=Sí, 0=No]: "))

# ── PERSONAL ─────────────────────────────────────────────────
age           = float(input("Edad al ingresar                 [ej: 20]: "))
displaced     = float(input("¿Desplazado?                     [1=Sí, 0=No]: "))
gender        = float(input("Género                           [1=Hombre, 0=Mujer]: "))

# ── MACROECONÓMICO ───────────────────────────────────────────
gdp           = float(input("GDP (tasa variación anual)       [ej: 1.74]: "))
unemployment  = float(input("Tasa de desempleo %              [ej: 10.8]: "))
inflation     = float(input("Tasa de inflación %              [ej: 1.4]: "))

# =============================
# FEATURE ENGINEERING AUTOMÁTICO
# =============================
aprobacion_rate_1 = cu1_approved / cu1_enrolled if cu1_enrolled > 0 else 0
aprobacion_rate_2 = cu2_approved / cu2_enrolled if cu2_enrolled > 0 else 0
variacion_rendimiento = cu2_grade - cu1_grade
carga_total = cu1_enrolled + cu2_enrolled
riesgo_financiero = int(tuition_ok == 0) + int(debtor == 1) + int(scholarship == 0)
ratio_notas = cu2_grade / (cu1_grade + 1e-5)
estres_academico = carga_total / (age + 1)

# =============================
# CONSTRUIR VECTOR DE FEATURES
# =============================
features_order = [
    'Curricular units 1st sem (approved)',
    'Curricular units 1st sem (grade)',
    'Curricular units 2nd sem (approved)',
    'Curricular units 2nd sem (grade)',
    'Tuition fees up to date',
    'Debtor',
    'Scholarship holder',
    'Age at enrollment',
    'Displaced',
    'Gender',
    'GDP',
    'Unemployment rate',
    'Inflation rate',
    'aprobacion_rate_1',
    'aprobacion_rate_2',
    'variacion_rendimiento',
    'carga_total',
    'riesgo_financiero',
    'ratio_notas',
    'estres_academico'
]

valores = [
    cu1_approved, cu1_grade,
    cu2_approved, cu2_grade,
    tuition_ok, debtor, scholarship,
    age, displaced, gender,
    gdp, unemployment, inflation,
    aprobacion_rate_1, aprobacion_rate_2,
    variacion_rendimiento, carga_total,
    riesgo_financiero, ratio_notas, estres_academico
]

X_nuevo = pd.DataFrame([valores], columns=features_order)

# =============================
# IMPUTAR + ESCALAR + PREDECIR
# =============================
# Usa el mismo imputer y scaler entrenados en Fase 4
X_nuevo_imp    = imputer.transform(X_nuevo)
X_nuevo_scaled = scaler.transform(X_nuevo_imp)

proba    = mlp.predict_proba(X_nuevo_scaled)[0, 1]
pred     = int(proba >= best_thr)

# =============================
# RESULTADO
# =============================
print()
print("=" * 60)
print("                    RESULTADO")
print("=" * 60)
print(f"  Probabilidad de deserción : {proba * 100:.1f}%")
print(f"  Threshold usado           : {best_thr:.2f}")
print()
if pred == 1:
    nivel = "ALTO" if proba > 0.70 else "MODERADO"
    print(f"  ⚠️  RIESGO {nivel} DE DESERCIÓN")
else:
    print(f"  ✅  ESTUDIANTE SIN RIESGO SIGNIFICATIVO")
print("=" * 60)

# =============================
# DETALLE DE FEATURES CALCULADAS
# =============================
print("\n── Features derivadas calculadas ──")
print(f"  Tasa aprobación 1er sem   : {aprobacion_rate_1:.2f}")
print(f"  Tasa aprobación 2do sem   : {aprobacion_rate_2:.2f}")
print(f"  Variación de rendimiento  : {variacion_rendimiento:+.2f}")
print(f"  Carga académica total     : {carga_total:.0f}")
print(f"  Índice riesgo financiero  : {riesgo_financiero}/3")
print(f"  Ratio de notas (2do/1ro)  : {ratio_notas:.2f}")
print(f"  Estrés académico          : {estres_academico:.2f}")